# Replicate Visual Search Asymmetries 
from Fig. 5.31 from "Understanding Vision", Zhaoping 2014

In [ ]:
import numpy as np 
import matplotlib.pyplot as plt

from v1sh_model.inputs.visualize import visualize_input, visualize_output
from v1sh_model.models.V1_model_1 import V1_model_1 as V1_model

# 1. Curved versus straight 

In [ ]:
def get_straight_and_curved_line(I_contrast = 3.5):
    C = np.zeros((2, 1))
    C[:, 0] = I_contrast
    A_straight = np.zeros((2, 1)) # vertical
    A_curved = np.zeros((2, 1))
    A_curved[0, 0] = 10 / 12 * np.pi
    A_curved[1, 0] = 2 / 12 * np.pi
    return (A_curved, C), (A_straight, C)

def generate_visual_search_input(more_salient_input, less_salient_input, 
                                 spatial_jitter_size = 1, 
                                 orientation_jitter_set = None, 
                                 size=(60, 60),
                                 seed = None,
                                 spacing = 1):
    
    A_in_1, C_in_1 = more_salient_input
    A_in_2, C_in_2 = less_salient_input
    
    assert C_in_1.shape == C_in_2.shape, "Input arrays must have the same shape"
    assert A_in_2.shape == A_in_1.shape, "Input arrays must have the same shape"
    assert A_in_1.shape == C_in_1.shape, "Input arrays must have the same shape"
    
    # 1. define grid points and target position
    x_fig, y_fig = C_in_1.shape
    
    delta_x = x_fig + 2 * spatial_jitter_size + spacing
    delta_y = y_fig + 2 * spatial_jitter_size + spacing
    size_x = (size[0] // delta_x) * delta_x # uniformize grid to avoid boundary artifacts 
    size_y = (size[1] // delta_y) * delta_y
    x_positions = np.arange(spatial_jitter_size, size_x, delta_x)
    y_positions = np.arange(spatial_jitter_size, size_y, delta_y)
    
    target_index_x = len(x_positions) // 2
    target_index_y = len(y_positions) // 2
    
    # 3. generate array with random jitter
    rng = np.random.default_rng(seed) 
    A_out_1, A_out_2 = np.zeros((size_x, size_y)), np.zeros((size_x, size_y))
    C_out_1, C_out_2 = np.zeros((size_x, size_y)), np.zeros((size_x, size_y))
    
    if spatial_jitter_size > 0:
        spatial_jitter_choices = np.arange(-spatial_jitter_size, spatial_jitter_size + 1)   
        spatial_jitter_choices = spatial_jitter_choices[spatial_jitter_choices != 0]  # exclude zero jitter
        
    target_positions = None
    for i, x in enumerate(x_positions):
        for j, y in enumerate(y_positions):
            if spatial_jitter_size == 0:
                x_jitter, y_jitter = 0, 0
            else:
                x_jitter, y_jitter = rng.choice(spatial_jitter_choices), rng.choice(spatial_jitter_choices)
            
            # TODO: add orientation jitter
            
            if not (i == target_index_x and j == target_index_y):
                A_out_1[x + x_jitter : x + x_jitter + x_fig, y + y_jitter : y + y_jitter + y_fig] = A_in_2
                C_out_1[x + x_jitter : x + x_jitter + x_fig, y + y_jitter : y + y_jitter + y_fig] = C_in_2
                
                A_out_2[x + x_jitter : x + x_jitter + x_fig, y + y_jitter : y + y_jitter + y_fig] = A_in_1
                C_out_2[x + x_jitter : x + x_jitter + x_fig, y + y_jitter : y + y_jitter + y_fig] = C_in_1
            else:
                A_out_1[x + x_jitter : x + x_jitter + x_fig, y + y_jitter : y + y_jitter + y_fig] = A_in_1
                C_out_1[x + x_jitter : x + x_jitter + x_fig, y + y_jitter : y + y_jitter + y_fig] = C_in_1
                
                A_out_2[x + x_jitter : x + x_jitter + x_fig, y + y_jitter : y + y_jitter + y_fig] = A_in_2
                C_out_2[x + x_jitter : x + x_jitter + x_fig, y + y_jitter : y + y_jitter + y_fig] = C_in_2

                target_positions = [x + x_jitter, x + x_jitter + x_fig, y + y_jitter, y + y_jitter + y_fig]
    
    assert target_positions is not None, "Target position was not set correctly."
    
    return (A_out_1, C_out_1), (A_out_2, C_out_2), target_positions

more_salient_input, less_salient_input = get_straight_and_curved_line(I_contrast = 3.5)
input_1, input_2, target_positions = generate_visual_search_input(more_salient_input, less_salient_input, spatial_jitter_size = 1, size=(60, 60), seed = None, spacing = 2)

A_1, C_1 = input_1
visualize_input(A_1, C_1, dpi = 200, verbose = False)
plt.title("More salient input (curved line among straight lines)")
plt.show()

A_2, C_2 = input_2
visualize_input(A_2, C_2, dpi = 200, verbose = False)
plt.title("Less salient input (straight line among curved lines)")
plt.show()

In [ ]:
T = 12.0
dt = 0.01
noisy = False
mode = "wrap"
model = V1_model()

X_out_1, _, _ = model.simulate(A_1, C_1, T = T, dt = dt, noisy = noisy, mode = mode)
X_out_2, _, _ = model.simulate(A_2, C_2, T = T, dt = dt, noisy = noisy, mode = mode)

In [ ]:
model_output_1 = model.g_x(X_out_1).mean(axis = 0)
model_output_2 = model.g_x(X_out_2).mean(axis = 0)
A_out = np.broadcast_to(model.M, model_output_1.shape)

visualize_output(A_out, model_output_1, scale = 0.3, dpi = 200, verbose = False)
plt.title("Output activity for more salient input")
plt.show()

visualize_output(A_out, model_output_2, scale = 0.3, dpi = 200, verbose = False)
plt.title("Output activity for less salient input") 
plt.show()

In [ ]:
def z_score(g_X, target_positions, cutoff = 0.1):
    g_X_temporal_average = np.mean(g_X, axis = 0)
    g_X_spatial_average = np.mean(g_X_temporal_average[g_X_temporal_average > cutoff])
    g_X_spatial_std = np.std(g_X_temporal_average[g_X_temporal_average > cutoff])
    SMAP = (g_X_temporal_average - g_X_spatial_average) / g_X_spatial_std
    z = np.max(SMAP[target_positions[0]:target_positions[1], target_positions[2]:target_positions[3]])
    return z

z_1 = z_score(model.g_x(X_out_1), target_positions)
z_2 = z_score(model.g_x(X_out_2), target_positions)

print(f"Z-score for more salient input: {z_1:.2f}")
print(f"Z-score for less salient input: {z_2:.2f}")

In [ ]:
N = 15
model_ouputs_1 = []
model_outputs_2 = []
target_outputs_1 = []
target_outputs_2 = []
z_1s = []
z_2s = []

for seed in np.arange(N): 
    input_1, input_2, target_positions = generate_visual_search_input(more_salient_input, less_salient_input, spatial_jitter_size = 1, size=(60, 60), seed = seed, spacing = 2)
    A_1, C_1 = input_1
    A_2, C_2 = input_2
   
    X_out_1, _, _ = model.simulate(A_1, C_1, T = T, dt = dt, noisy = noisy, mode = mode)
    X_out_2, _, _ = model.simulate(A_2, C_2, T = T, dt = dt, noisy = noisy, mode = mode)
    
    model_output_1 = model.g_x(X_out_1).mean(axis = 0)
    model_output_2 = model.g_x(X_out_2).mean(axis = 0)
    model_ouputs_1.append(model_output_1[model_output_1 > 0].flatten())
    model_outputs_2.append(model_output_2[model_output_2 > 0].flatten())
    
    target_outputs_1.append(np.max(model_output_1[target_positions[0]:target_positions[1], target_positions[2]:target_positions[3]]))
    target_outputs_2.append(np.max(model_output_2[target_positions[0]:target_positions[1], target_positions[2]:target_positions[3]]))
    
    z_1 = z_score(model.g_x(X_out_1), target_positions, cutoff = 0.3)
    z_2 = z_score(model.g_x(X_out_2), target_positions, cutoff = 0.3)
    z_1s.append(z_1)
    z_2s.append(z_2)

In [ ]:
z_1_mean = np.mean(z_1s)
z_2_mean = np.mean(z_2s)
z_1_sem = np.std(z_1s) / np.sqrt(N)
z_2_sem = np.std(z_2s) / np.sqrt(N)

print(f"Z-score for more salient input: {z_1_mean:.2f} ± {z_1_sem:.2f}")
print(f"Z-score for less salient input: {z_2_mean:.2f} ± {z_2_sem:.2f}")    

In [ ]:
plt.title("Easier search (more salient target)")
plt.hist(np.concatenate(model_ouputs_1), bins = 100, label = "Background", color = "blue")
plt.hist(np.array(target_outputs_1), bins = 20, label = "Target", color = 'red')
plt.xlim(0.5, 1)
plt.yscale('log')
plt.legend()
plt.show()

plt.title("Harder search (less salient target)")
plt.hist(np.concatenate(model_outputs_2), bins = 100, label = "Background", color = "blue")
plt.hist(np.array(target_outputs_2), bins = 20, label = "Target", color = 'red')
plt.xlim(0.5, 1)
plt.yscale('log')
plt.legend()
plt.show()